# 🚀 Advanced PySpark Performance Tuning & Optimization Notebook
This interactive notebook pairs production-grade PySpark code examples with deep architectural explanations, focusing on execution plans, Catalyst optimization, memory handling, AQE, and join strategies.

--- 
## 1. Predicate Pushdown & Storage Skipping (Parquet)
### **Concept & Explanation**
* **Physical Plan Indicator:** Look for `PushedFilters: [IsNotNull(age), GreaterThan(age,30)]` and `Batched: true` inside the `FileScan` operator.
* **Why it's optimized:** Parquet stores data columnarly along with file footers containing min/max statistics for every row group. When you apply a filter on a Parquet read, Spark passes the condition directly to the storage reader.
* **Production Best Practice:** Always apply filters early in your pipeline. Skipping non-matching row groups directly at the storage layer prevents unnecessary disk read and network I/O.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Optimization_1_Predicate_Pushdown") \
    .master("local[*]") \
    .getOrCreate()

# Step 1: Create and write sample dataset to Parquet
data = [(i, f"user_{i}", i % 80) for i in range(100000)]
df = spark.createDataFrame(data, ["id", "name", "age"])
df.write.mode("overwrite").parquet("/tmp/parquet_pushdown_demo")

# Step 2: Read back and filter early
df_read = spark.read.parquet("/tmp/parquet_pushdown_demo")
filtered_df = df_read.filter("age > 50")

# Step 3: Inspect execution plan
print("=== Physical Plan: Parquet Predicate Pushdown ===")
filtered_df.explain(True)

--- 
## 2. Catalyst Optimizer Reordering & WholeStageCodegen
### **Concept & Explanation**
* **Physical Plan Indicator:** Look for stage identifiers like `*(1) Project` and `*(1) Filter` containing asterisks (`*`).
* **Why it's optimized:** The Catalyst Optimizer evaluates the execution tree and dynamically reorders operations into an optimal logical structure. The `*(1)` operator confirms that **WholeStageCodegen** (Tungsten Engine) is active, fusing multiple operators into a single Java bytecode execution loop to maximize CPU L1/L2 cache efficiency.
* **Production Best Practice:** Write clean, expressive PySpark code using standard native expressions so Catalyst can recognize pushdown and projection opportunities.

In [ ]:
print("=== Demonstration: WholeStageCodegen & Catalyst Optimization ===")
df_range = spark.range(0, 1000000) \
    .withColumn("val", F.col("id") * 2) \
    .filter("val > 500000")

df_range.explain(True)

--- 
## 3. Broadcast Hash Join vs. Sort-Merge Join
### **Concept & Explanation**
* **Physical Plan Indicator:** Look for `BroadcastHashJoin` and `BroadcastExchange` replacing `SortMergeJoin` and `Exchange hashpartitioning`.
* **Why it's optimized:** In a standard join, Spark shuffles both tables across the network. If one table is small (default threshold: 10MB via `spark.sql.autoBroadcastJoinThreshold`), Spark copies the entire small table to every executor worker, allowing the join to finish in local memory with zero network shuffles.
* **Production Best Practice:** Explicitly hint small lookup/dimension tables with `F.broadcast()` to eliminate heavy shuffles.

In [ ]:
# Generate a large dataset and a small lookup dimension
large_df = spark.range(0, 500000).withColumn("key", F.col("id") % 100)
small_df = spark.range(0, 50).withColumn("key", F.col("id")).withColumn("category", F.lit("Tier_1"))

print("=== Physical Plan: Broadcast Hash Join ===")
broadcast_joined = large_df.join(F.broadcast(small_df), "key")
broadcast_joined.explain(True)

--- 
## 4. Adaptive Query Execution (AQE) Data Skew Mitigation
### **Concept & Explanation**
* **Physical Plan Indicator:** Inspect plan output post-action execution for `CustomShuffleReader (skewed)` or `CustomShuffleReader (coalesced)`.
* **Why it's optimized:** Data skew (e.g., 70% of transactions sharing a single hot key like `customer_id = 0`) forces one worker task to process almost all data while others sit idle. With AQE enabled (`spark.sql.adaptive.enabled=true`), Spark detects bottlenecked partitions at runtime, automatically splitting them into sub-tasks.
* **Production Best Practice:** Always enable AQE flags in enterprise pipelines as an automated safety layer against skew and small file/partition overhead.

In [ ]:
# Enable AQE configurations
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1") # Force Sort-Merge Join to demonstrate skew handling

# Create severe data skew (70% hot key = 0)
txns = spark.range(0, 500000).withColumn("customer_id", F.when(F.rand() < 0.7, 0).otherwise((F.rand() * 1000).cast("int")))
customers = spark.range(0, 1000).withColumn("customer_id", F.col("id"))

skewed_join = txns.join(customers, "customer_id", "inner")
count = skewed_join.count() # Action populates runtime statistics

print("=== Physical Plan: AQE Post-Execution Skew Plan ===")
skewed_join.explain(True)

--- 
## 5. Standard Python UDFs vs. Apache Arrow Vectorized Pandas UDFs
### **Concept & Explanation**
* **Why standard UDFs are slow:** Standard `@udf` functions require row-by-row data serialization out of the JVM into a Python worker process, completely bypassing WholeStageCodegen.
* **Why Vectorized/Native expressions are fast:** Native PySpark functions run entirely inside the JVM. If custom Python logic is mandatory, `@pandas_udf` utilizes Apache Arrow for vectorized batch memory transfer between the JVM and Python.
* **Production Best Practice:** Avoid standard Python UDFs. Prioritize Native PySpark expressions first, followed by Apache Arrow Vectorized Pandas UDFs.

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd
import time

# Sample Data
df_bench = spark.range(0, 1000000).withColumn("v1", F.rand()).withColumn("v2", F.rand()).cache()
df_bench.count()

# 1. Standard Python UDF (Row-by-Row)
@udf(returnType=DoubleType())
def slow_python_udf(v1, v2):
    return float(v1 * 0.5 + v2 * 1.5)

start = time.time()
df_bench.withColumn("res", slow_python_udf("v1", "v2")).agg(F.sum("res")).collect()
print(f"Standard Python UDF Time: {time.time() - start:.2f}s")

# 2. Vectorized Pandas UDF (Apache Arrow)
@pandas_udf(DoubleType())
def vectorized_pandas_udf(v1: pd.Series, v2: pd.Series) -> pd.Series:
    return v1 * 0.5 + v2 * 1.5

start = time.time()
df_bench.withColumn("res", vectorized_pandas_udf("v1", "v2")).agg(F.sum("res")).collect()
print(f"Vectorized Pandas UDF Time: {time.time() - start:.2f}s")

# 3. Native PySpark Expression (Best)
start = time.time()
df_bench.withColumn("res", F.col("v1") * 0.5 + F.col("v2") * 1.5).agg(F.sum("res")).collect()
print(f"Native PySpark Expression Time: {time.time() - start:.2f}s")

spark.stop()